In [1]:
# 获取大模型
import os
from langchain_openai import ChatOpenAI
import dotenv

dotenv.load_dotenv(override=True)
prefix = "DAIPAI_"
model_name = os.getenv(prefix + "MODEL")
model_base_url = os.getenv(prefix + "BASE_URL")
model_api_key = os.getenv(prefix + "API_KEY")

print(f"Model Name: {model_name}")


chat_llm = ChatOpenAI(model=model_name or "gpt-3.5-turbo", base_url=model_base_url, api_key=model_api_key) # type: ignore

Model Name: daipai/qwen/qwen3-next-80b-a3b-thinking


In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough


# 方式1：最简单的顺序链（单输入单输出）
# 第一步：根据标题生成剧情梗概
synopsis_prompt = ChatPromptTemplate.from_template(
    "你是剧作家。根据给定的戏剧标题，为该剧写一篇剧情梗概。\n\n标题：{title}\n\n剧情梗概："
)
# 第二步：根据梗概写剧评
review_prompt = ChatPromptTemplate.from_template(
    "你是《纽约时报》的剧评家。根据以下剧情梗概，为该剧写一篇评论。\n\n剧情梗概：\n{synopsis}\n\n评论："
)

# 用管道符串联起来
# 构建链并保留中间结果
chain = (
    # 第一步：生成梗概
    RunnableParallel(
        title=RunnablePassthrough(),
        synopsis=(
            synopsis_prompt 
            | chat_llm 
            | StrOutputParser()
        )
    )
    # 第二步：使用title和synopsis生成评论
    | RunnableParallel(
        synopsis=lambda x: x["synopsis"],
        review=(
            review_prompt 
            | chat_llm 
            | StrOutputParser()
        )
    )
)


# 执行
result = chain.invoke({"title": "西工大十年风雪史"})
print(result)

{'synopsis': '\n\n## 西工大十年风雪史\n\n**人物：**  \n- **陈明远**：50岁，西北工业大学航空发动机研究所教授，项目负责人，清瘦坚韧，袖口磨出毛边的旧棉袄是他的标志。  \n- **林薇**：25岁，博士生，陈明远的得意门生，眼神清澈却带着倔强，笔记本扉页写着“风雪夜归人”。  \n- **李强**：32岁，陈明远的前助教，后跳槽至某民营航空企业，西装笔挺，但眼底有未熄灭的火焰。  \n- **赵国栋**：55岁，副校长，西装革履，言谈间总带着“大局为重”的谨慎，抽屉里压着一叠未批的项目续费申请。  \n\n---\n\n### 第一幕：寒夜断弦（2010年冬）  \n西北工业大学的实验室窗上结满冰花。陈明远独自坐在昏黄的灯光下，面前摊着一份“项目终止通知书”。窗外，大雪封门，风卷着雪粒抽打玻璃。他抚摸着桌上一台老旧的发动机模型，指尖划过冰冷的金属——这是他十年心血的结晶。  \n\n赵国栋推门而入，语气沉重：“明远，经费砍了，人手撤了。学校要集中资源保‘双一流’评估，这种‘高投入、低产出’的项目……”陈明远没有抬头，只是将模型轻轻放回抽屉，抽屉里还压着十年前他写给上级的报告：“此项目若成功，将改变中国航空史。”赵国栋瞥见报告，喉头动了动，最终只留下一句“等风雪过去再说”，转身离去。  \n\n林薇抱着几本资料冲进实验室，头发上沾着雪：“陈老师，我刚从北京回来！国家新出了重大专项，要求十年内突破航空发动机核心部件技术！我们……”话音未落，陈明远却只是将通知书推到她面前。林薇看着“终止”二字，攥紧的拳头微微发抖。窗外，雪越下越大，仿佛要将整个校园埋葬。  \n\n---\n\n### 第二幕：孤灯拾薪（2012年春）  \n实验室角落堆着被遗弃的设备，蒙着灰。陈明远在雪夜中跋涉，将一台被当作废品处理的精密仪器拖回实验室。他用旧棉袄裹住仪器，呵出的白气在寒风中凝成霜。  \n\n林薇提着一盏煤油灯走进来：“老师，我查到了——国家专项的申报窗口还有三个月。”陈明远看着灯芯摇曳的微光，忽然笑了：“好，那就用这盏灯，照亮剩下的路。”两人在雪夜中清理设备，煤油灯映照下，陈明远的棉袄袖口又裂开一道口子。  \n\n赵国栋在办公室接到电话：“陈教授又申请了经费？他连实验室都快维持不下去了！”他翻出抽屉里那张“改变中国航空史”的报告，指尖在“十年

In [7]:
print(result["review"])
print(result["synopsis"])



## 风雪中的灯：《西工大十年风雪史》如何重塑中国科技叙事

当陈明远教授用磨破袖口的旧棉袄裹住被遗弃的精密仪器时，他不仅在拯救一台设备，更在守护一种精神——这种精神在当今世界科技竞争中比任何专利都更珍贵。《西工大十年风雪史》这部中国原创话剧，以西北工业大学航空发动机研究所的十年攻坚史为蓝本，用五个冬春交替的场景，完成了一次对“中国科技叙事”的深刻重构。它拒绝将国家成就简化为宏大口号，而是将镜头对准风雪夜中那些不肯熄灭的灯，让世界看到：真正的科技突破，从来不是冰冷的国家意志，而是无数个体在绝望中点燃的微光。

**风雪作为隐喻的精妙运用，是该剧最震撼人心的艺术成就。** 开篇“寒夜断弦”中，实验室窗上结满的冰花不仅是西北严寒的写实，更是中国科研工作者面临的无形重压。当赵国栋副校长说出“这种‘高投入、低产出’的项目”时，他手中“未批的项目续费申请”与抽屉里“改变中国航空史”的旧报告形成残酷对照——这种体制内现实与理想主义的撕裂，被具象化为雪夜中颤抖的笔尖。而“孤灯拾薪”一幕中，林薇提着的煤油灯摇曳的光晕，不仅照亮了被遗弃的设备，更照亮了中国科研史上那些被主流叙事忽略的角落：没有“大跃进”式的狂热，只有“用这盏灯，照亮剩下的路”的朴素坚持。

**人物塑造的深度，让这部剧超越了简单的“主旋律”框架。** 陈明远的旧棉袄袖口磨出毛边，这个细节比任何台词都更具力量——它不是刻意的悲情渲染，而是十年如一日的坚守刻在衣物上的年轮。当林薇将笔记本扉页从“风雪夜归人”改为“风雪长明”时，她完成的不仅是个人成长，更是对“科研精神”的重新定义：真正的“归人”不是抵达终点的旅人，而是始终在风雪中点灯的守灯者。更值得称道的是李强这个角色——西装革履的民营企业家，撕碎支票时的沉默，比任何慷慨陈词都更有说服力。他眼底“未熄灭的火焰”最终选择留在实验室，恰恰证明：当国家力量与个人理想达成共振时，科技突破才真正成为可能。

**剧作的最高成就，在于它颠覆了西方对中国科技发展的刻板认知。** 在《纽约时报》等西方媒体的叙事中，中国科技成就常被简化为“国家投入”“举国体制”等宏大概念。但《西工大十年风雪史》用赵国栋这个角色揭示了真相：当他在“冰封时刻”看到实验台下堆满的失败记录——“每一页都写满公式，每一页都沾着油污和汗渍”时，他最终签下的“再给三个月”，不是来自上级命令，而是对人类科学精神的敬畏

In [9]:
from langchain_core.runnables import RunnableLambda
parser = StrOutputParser()

# 第一步：回答问题1 —— 剧情梗概
synopsis_prompt = ChatPromptTemplate.from_template(
    "你是剧作家。根据标题写一段剧情梗概。\n\n标题：{title}\n\n剧情梗概："
)

synopsis_chain = synopsis_prompt | chat_llm | parser

# 第二步：回答问题2 —— 主题提炼（依赖第一步）
theme_prompt = ChatPromptTemplate.from_template(
    "你是文学评论家。根据下面的剧情梗概，提炼这部戏的核心主题。\n\n"
    "剧情梗概：\n{synopsis}\n\n"
    "核心主题："
)

theme_chain = theme_prompt | chat_llm | parser

# 第三步：回答问题3 —— 最终分析（依赖前两步）
final_prompt = ChatPromptTemplate.from_template(
    "你是资深戏剧研究者。请根据以下信息，写一段综合分析。\n\n"
    "剧情梗概：\n{synopsis}\n\n"
    "核心主题：\n{theme}\n\n"
    "请回答：这部戏为什么能打动观众？"
)

# 先拿到 synopsis，再基于 synopsis 同时生成 synopsis 和 theme，
# 最后把它们一起传给 final_prompt
chain = (
    synopsis_chain
    | RunnableLambda(
        lambda synopsis: {
            "synopsis": synopsis,
            "theme": theme_chain.invoke({"synopsis": synopsis}),
        }
    )
    | final_prompt
    | chat_llm
    | parser
)

res = chain.invoke({"title": "哈姆雷特"})
print(res)



## 《哈姆雷特》何以穿越时空打动人心：在毁灭的深渊中照见人性的微光

作为一部诞生于1601年的戏剧，《哈姆雷特》历经四百年仍能令全球观众潸然泪下、陷入沉思，其核心魅力正在于莎士比亚以惊人的艺术洞察力，将人类最根本的生存困境具象化为一场血色宫廷悲剧。这种打动并非来自简单的善恶二分或复仇快感，而在于它精准地刺中了每个现代灵魂深处的隐痛。

### 一、**“犹豫”背后的普遍人性共鸣**
哈姆雷特绝非传统英雄。他面对弑父之仇时的反复踌躇、对行动的自我怀疑，恰恰打破了“复仇者必果决”的刻板叙事。当他说出“生存还是毁灭”时，观众看到的不仅是王子的困境，更是每个在道德十字路口徘徊的普通人——我们是否该为受辱反击？是否该为正义冒险？这种“行动的瘫痪”在当代社会尤为刺痛：面对网络暴力、职场不公、社会不义时，我们何尝不是哈姆雷特？莎士比亚没有给出答案，却将“犹豫”本身升华为一种高贵的人性真实。当哈姆雷特在舞台上演绎着“该不该杀克劳狄斯”的内心挣扎，观众席上的每个人都在镜像中看见自己的道德困境。

### 二、**“疯癫”的双重隐喻：清醒者与被压迫者的双重悲剧**
哈姆雷特的“疯癫”是精妙的戏剧装置：表面是装疯卖傻的策略，内里却是清醒者对荒诞世界的绝望控诉。当他说“丹麦是座监狱”时，观众瞬间理解了他为何要“装疯”——因为清醒地活着才是真正的疯狂。而奥菲莉亚的真疯与溺亡，更将这种悲剧推向极致：一个被父权、爱情、政治多重压迫的女性，最终只能以死亡获得解脱。这种“疯癫”叙事让现代观众看到：在权力结构中，所谓“正常”往往意味着对压迫的默许，而“疯”反而是最后的抵抗。当奥菲莉亚在舞台上唱着破碎的歌谣漂向死亡，无数女性观众会想起自己或身边人被社会规训逼至崩溃的瞬间。

### 三、**“复仇徒劳”的哲学震撼：暴力循环的现代寓言**
克劳狄斯的毒药灌耳、哈姆雷特的误杀波洛涅斯、雷欧提斯的毒剑复仇、王后误饮毒酒——这些死亡如多米诺骨牌般连锁爆发，构成对“以暴制暴”最残酷的解构。莎士比亚没有美化复仇，反而展示其本质：**复仇不是正义的伸张，而是暴力的病毒式传染**。当哈姆雷特最终刺死克劳狄斯时，他已失去所有珍视之人，连复仇本身都沦为自我毁灭的工具。这与当代社会的“冤冤相报”何其相似：恐怖袭击引发的报复性战争、网络暴力引发的群体性围剿，无不印证着“复仇吞噬复仇者”的真理。观众在剧场中看到的不仅是1

In [10]:
synopsis_prompt = ChatPromptTemplate.from_template(
    "根据标题写剧情梗概。\n标题：{title}\n剧情梗概："
)

theme_prompt = ChatPromptTemplate.from_template(
    "根据下面的剧情梗概，提炼核心主题。\n\n剧情梗概：{synopsis}\n\n核心主题："
)

final_prompt = ChatPromptTemplate.from_template(
    "请根据以下信息回答：这部戏为什么能打动观众？\n\n"
    "标题：{title}\n"
    "剧情梗概：{synopsis}\n"
    "核心主题：{theme}\n"
)

synopsis_chain = synopsis_prompt | chat_llm | parser
theme_chain = theme_prompt | chat_llm | parser
final_chain = final_prompt | chat_llm | parser

chain = (
    RunnablePassthrough()
    .assign(synopsis=synopsis_chain)
    .assign(theme=theme_chain)
    .assign(final_answer=final_chain)
)

result = chain.invoke({"title": "哈姆雷特"})
print(result)

{'title': '哈姆雷特', 'synopsis': '\n\n**《哈姆雷特》剧情梗概**  \n\n丹麦王子哈姆雷特的父亲——老国王突然离世，其叔父克劳狄斯迅速继位并娶走王后乔特鲁德。哈姆雷特悲愤难平，其父的鬼魂现身，揭露克劳狄斯毒杀自己篡位的真相，并命他复仇。为试探真相，哈姆雷特伪装疯狂，借“戏中戏”证实叔父的罪行。  \n\n误杀御前大臣波洛涅斯后，克劳狄斯借机将哈姆雷特送往英国欲除之，却反被哈姆雷特识破阴谋。波洛涅斯之女奥菲莉亚因爱而疯，溺水身亡，其兄雷欧提斯誓要复仇。克劳狄斯设计毒剑与毒酒，挑拨雷欧提斯与哈姆雷特决斗。决斗中，王后误饮毒酒身亡，雷欧提斯与哈姆雷特均中剧毒剑伤。临死前，哈姆雷特刺死克劳狄斯，托付好友霍拉旭将真相传世，自己含恨离世。', 'theme': '\n\n**核心主题：复仇的道德困境与毁灭性**  \n\n《哈姆雷特》通过哈姆雷特的复仇之旅，深刻探讨了人性在正义与道德之间的撕裂：  \n- **道德困境**：哈姆雷特明知叔父克劳狄斯弑君篡位的罪行，却在行动前反复质疑复仇的正当性（如“生存还是毁灭”的独白）。他拒绝在克劳狄斯祈祷时下手，认为杀死忏悔的灵魂会使其升入天堂，反而违背“正义”；但若等待对方犯下新罪再复仇，又可能错失时机。这种对“复仇是否合乎道德”的纠结，揭示了个人良知与社会复仇法则的冲突。  \n- **毁灭性后果**：复仇并非简单的“正义伸张”，而是引发连锁悲剧。哈姆雷特的犹豫导致误杀波洛涅斯、奥菲莉亚发疯溺亡；克劳狄斯的阴谋毒害王后、雷欧提斯；最终哈姆雷特虽刺死仇敌，自己也身中剧毒而亡。整个丹麦宫廷在复仇中走向覆灭，证明“以暴制暴”不仅无法修复创伤，反而将所有人拖入更深的毁灭。  \n\n这一主题超越了个人恩怨，直指人类面对不公时的普遍困境：当“正义”需要通过暴力实现，行动的代价是否值得？真相的揭露是否必然导向救赎？莎士比亚通过悲剧结局警示——复仇的本质是自我吞噬，而真正的救赎或许在于对生命意义的思考（如哈姆雷特临终托付“真相传世”），而非单纯以血还血。', 'final_answer': '\n\n《哈姆雷特》之所以能跨越400年仍深深打动观众，核心在于它以悲剧的棱镜折射出人类永恒的精神困境。以下从三个维度解析其打动人心的深层原因：\n\n---\n\n### **一、道德困境的「真实感」：让每个观众都成为「哈姆

In [11]:
print(result["title"])
print(result["synopsis"])
print(result["theme"])
print(result["final_answer"])

哈姆雷特


**《哈姆雷特》剧情梗概**  

丹麦王子哈姆雷特的父亲——老国王突然离世，其叔父克劳狄斯迅速继位并娶走王后乔特鲁德。哈姆雷特悲愤难平，其父的鬼魂现身，揭露克劳狄斯毒杀自己篡位的真相，并命他复仇。为试探真相，哈姆雷特伪装疯狂，借“戏中戏”证实叔父的罪行。  

误杀御前大臣波洛涅斯后，克劳狄斯借机将哈姆雷特送往英国欲除之，却反被哈姆雷特识破阴谋。波洛涅斯之女奥菲莉亚因爱而疯，溺水身亡，其兄雷欧提斯誓要复仇。克劳狄斯设计毒剑与毒酒，挑拨雷欧提斯与哈姆雷特决斗。决斗中，王后误饮毒酒身亡，雷欧提斯与哈姆雷特均中剧毒剑伤。临死前，哈姆雷特刺死克劳狄斯，托付好友霍拉旭将真相传世，自己含恨离世。


**核心主题：复仇的道德困境与毁灭性**  

《哈姆雷特》通过哈姆雷特的复仇之旅，深刻探讨了人性在正义与道德之间的撕裂：  
- **道德困境**：哈姆雷特明知叔父克劳狄斯弑君篡位的罪行，却在行动前反复质疑复仇的正当性（如“生存还是毁灭”的独白）。他拒绝在克劳狄斯祈祷时下手，认为杀死忏悔的灵魂会使其升入天堂，反而违背“正义”；但若等待对方犯下新罪再复仇，又可能错失时机。这种对“复仇是否合乎道德”的纠结，揭示了个人良知与社会复仇法则的冲突。  
- **毁灭性后果**：复仇并非简单的“正义伸张”，而是引发连锁悲剧。哈姆雷特的犹豫导致误杀波洛涅斯、奥菲莉亚发疯溺亡；克劳狄斯的阴谋毒害王后、雷欧提斯；最终哈姆雷特虽刺死仇敌，自己也身中剧毒而亡。整个丹麦宫廷在复仇中走向覆灭，证明“以暴制暴”不仅无法修复创伤，反而将所有人拖入更深的毁灭。  

这一主题超越了个人恩怨，直指人类面对不公时的普遍困境：当“正义”需要通过暴力实现，行动的代价是否值得？真相的揭露是否必然导向救赎？莎士比亚通过悲剧结局警示——复仇的本质是自我吞噬，而真正的救赎或许在于对生命意义的思考（如哈姆雷特临终托付“真相传世”），而非单纯以血还血。


《哈姆雷特》之所以能跨越400年仍深深打动观众，核心在于它以悲剧的棱镜折射出人类永恒的精神困境。以下从三个维度解析其打动人心的深层原因：

---

### **一、道德困境的「真实感」：让每个观众都成为「哈姆雷特」**
- **「生存还是毁灭」的普世拷问**  
  哈姆雷特的犹豫并非懦弱，而是对复仇正义性的深度思辨。当他说「一个罪人带着他全部的罪恶升天」时，

In [14]:
joke_chain = (
    ChatPromptTemplate.from_template("tell me a joke about {topic}") | chat_llm
)
poem_chain = (
    ChatPromptTemplate.from_template("write a 2-line poem about {topic}")
    | chat_llm
)

runnable = RunnableParallel(joke=joke_chain, poem=poem_chain)

# Display stream
output = {"joke": "", "poem": ""}
for chunk in runnable.stream({"topic": "bear"}):
    for key in chunk:
        output[key] = output[key] + chunk[key].content
    if output["joke"] and output["poem"]:
        print(output)  # noqa: T201

{'joke': '\n\n', 'poem': '\n\nMighty shadow, forest deep,  \nPaws like stones crushing stones asleep.'}
{'joke': '\n\nWhat do you call a bear with no teeth?', 'poem': '\n\nMighty shadow, forest deep,  \nPaws like stones crushing stones asleep.'}
{'joke': '\n\nWhat do you call a bear with no teeth?  \n**A gummy bear!**', 'poem': '\n\nMighty shadow, forest deep,  \nPaws like stones crushing stones asleep.'}
{'joke': '\n\nWhat do you call a bear with no teeth?  \n**A gummy bear!** 🐻🍬  \n\n*(Pun alert: Gummy', 'poem': '\n\nMighty shadow, forest deep,  \nPaws like stones crushing stones asleep.'}
{'joke': '\n\nWhat do you call a bear with no teeth?  \n**A gummy bear!** 🐻🍬  \n\n*(Pun alert: Gummy bears are candy, but a toothless bear would', 'poem': '\n\nMighty shadow, forest deep,  \nPaws like stones crushing stones asleep.'}
{'joke': '\n\nWhat do you call a bear with no teeth?  \n**A gummy bear!** 🐻🍬  \n\n*(Pun alert: Gummy bears are candy, but a toothless bear would just have gums... "gum